## Map patches from PDB structure onto AF2 patches

This notebook demonstrates a proof-of-concept workflow to sample positive patch pairs from AF2 structures in the Pinder database. 

Purpose:
- To retrain MaSIF using AF2 models in the pinder database, we need to generate positive patch pairs from the AF2 models.
- For a given PDB, the pinder repo has already written the structures to the following files, where AF2 models are aligned to the same pose as in the corresponding chain of the PDB structure:
    - `1ATP-PP_L_R.pdb` - Both chain L and chain R are PDB structures
    - `1ATP-PA_L_R.pdb` - Chain L is PDB structure, chain R is AF2 structure 
    - `1ATP-AP_L_R.pdb` - Chain L is AF2 structure, chain R is PDB structure
    - `1ATP-AA_L_R.pdb` - Both chain L and chain R are AF2 structures
- The original positive sampling strategy works poorly on AF2 models, because it relies on a hard sc > 0.5 filter (shape complementarity), and Af2 models are expected to have poorer geometric complementarity. As a result, the original workflow samples very few positive pairs from AF2 models.

Proposed solution:
- For a given PDB id, use the original sampling strategy on the `PP` (PDB-PDB) complex in the pinder database first.
- Then, map the patches from the `PP` complex onto the AF2 models on their respective chains:
    - By aligning the AF2 model onto PDB structure (already done in Pinder) or by aligning the surface point cloud? To be figured out
    - For a given patch in PDB, find the nearest neighbour in the corresponding AF2 model based on centre coordinates. 
- Build the equivalent pairs as positives for the `AA`,`PA`,`AP` structures in the pinder database.

In [ ]:
# Assume you have made a symbolic link of pinderMaSIF repo under the repo root
import os
import sys
from pathlib import Path

repo_root = !git rev-parse --show-toplevel
repo_root = repo_root[0]
MASIF_PPI_SEARCH_DIR = Path(repo_root) / "masif/data/masif_ppi_search"

# Paths in pinderMaSIF repo
pinderMaSIF_repo_root = os.path.join(repo_root, "pinderMaSIF")
pinder_pdb_dir = os.path.join(pinderMaSIF_repo_root, "output", "pinder_filtered_260205", "pdbs")

# Directory to save full_list.txt
full_list_dir = os.path.join(repo_root, "masif/data/masif_ppi_search/lists")
os.makedirs(full_list_dir, exist_ok=True)

In [ ]:
# Check the format of the current cache catalog .npy files
import os
import sys
import numpy as np

cache_catalog_dir = MASIF_PPI_SEARCH_DIR / "nn_models/sc05/cache/model_data_cross_1to1/catalog"

records = np.load(cache_catalog_dir / "records.npy", allow_pickle=True)


In [ ]:
records[3]

___
### Step 0 - Read full_list.txt for example training data

In [ ]:
full_list_path = os.path.join(repo_root, "pinderMaSIF/output/pinder_filtered_260205/full_list.txt")
with open(full_list_path, "r") as f:
    full_list = [line.strip() for line in f if line.strip()]

# small list to test
small_list = full_list[1:6]
small_list

___
### Step 1 - Run data_prepare for a set of models (PP, PA, AP, AA)

In [ ]:
# Run data_prepare for all PP model (regardless of model type used in training)
# because PP model is necessary for mapping patches from PDB to AF2 models
small_list_pdb_accn = [pdb_id.split("-")[0] for pdb_id in small_list]

# Function to submit data_prepare_single.slurm for a given complex
def submit_data_prepare_single(complex_id):
    !sbatch data_prepare_single.slurm $complex_id $pinder_pdb_dir

# Run data_prepare for all PP models
for pdb_accn in small_list_pdb_accn:
    submit_data_prepare_single(f"{pdb_accn}-PP_L_R")


In [ ]:
# Then, run data_prepare for all the complexes in small_list (PP will be automatically skipped)
for complex_id in small_list:
    submit_data_prepare_single(complex_id)

___
### Step 2 - Apply the positive sampling strategy from build_cache_catalog to build a record for only the PP structure

In [ ]:
from pathlib import Path

import numpy as np
import py3Dmol
from scipy.spatial import cKDTree
from Bio.PDB import PDBParser, PDBIO

# Shared paths/config used by Steps 2-6.
REPO_ROOT = Path(repo_root)
PRECOMP_DIR = MASIF_PPI_SEARCH_DIR / "data_preparation/04b-precomputation_12A/precomputation"
PLY_DIR = MASIF_PPI_SEARCH_DIR / "data_preparation/01-benchmark_surfaces"
PINDER_PDB_DIR = Path(pinder_pdb_dir)

POS_SC_MIN = 0.5
POS_SC_MAX = 1.0
POS_INTERFACE_CUTOFF = 1.0


def load_vertices_from_ply(ply_fn):
    """PyMesh-free .ply vertex loader (trimesh first, plyfile fallback)."""
    try:
        import trimesh

        mesh = trimesh.load_mesh(str(ply_fn), process=False)
        vertices = np.asarray(mesh.vertices)
        if vertices.ndim == 2 and vertices.shape[1] == 3:
            return vertices
    except Exception:
        pass

    try:
        from plyfile import PlyData

        ply = PlyData.read(str(ply_fn))
        v = ply["vertex"]
        return np.column_stack([v["x"], v["y"], v["z"]]).astype(float)
    except Exception as exc:
        raise RuntimeError(
            "Could not load PLY vertices. Install trimesh or plyfile in this kernel."
        ) from exc


def parse_ppi_pair_id(ppi_pair_id):
    fields = ppi_pair_id.split("_")
    if len(fields) < 3:
        raise ValueError(f"Invalid ppi_pair_id format: {ppi_pair_id}")
    return fields[0], fields[1], fields[2]


def assert_inputs_exist(ppi_pair_id):
    pdb_id, ch1, ch2 = parse_ppi_pair_id(ppi_pair_id)
    required = [
        PRECOMP_DIR / ppi_pair_id,
        PRECOMP_DIR / ppi_pair_id / "p1_sc_labels.npy",
        PLY_DIR / f"{pdb_id}_{ch1}.ply",
        PLY_DIR / f"{pdb_id}_{ch2}.ply",
        PINDER_PDB_DIR / f"{ppi_pair_id}.pdb",
    ]
    missing = [str(p) for p in required if not p.exists()]
    if missing:
        raise FileNotFoundError("Missing required inputs:\n" + "\n".join(missing))


def build_pp_record(ppi_pair_id, split="train", sc_min=POS_SC_MIN, sc_max=POS_SC_MAX, interface_cutoff=POS_INTERFACE_CUTOFF):
    """Mirror build_cache_catalog positive sampling for one PP complex."""
    assert_inputs_exist(ppi_pair_id)
    pdb_id, ch1, ch2 = parse_ppi_pair_id(ppi_pair_id)

    in_dir = PRECOMP_DIR / ppi_pair_id
    labels = np.load(in_dir / "p1_sc_labels.npy")
    labels = np.median(labels[0], axis=1)

    pos_labels = np.where((labels < sc_max) & (labels > sc_min))[0]
    if len(pos_labels) == 0:
        raise RuntimeError(f"No SC-filtered positives for {ppi_pair_id}")

    v1_all = load_vertices_from_ply(PLY_DIR / f"{pdb_id}_{ch1}.ply")
    v2_all = load_vertices_from_ply(PLY_DIR / f"{pdb_id}_{ch2}.ply")

    v1_candidates = v1_all[pos_labels]
    kdt_v2 = cKDTree(v2_all)
    d_contact, r_contact = kdt_v2.query(v1_candidates)
    contact_points = np.where(d_contact < interface_cutoff)[0]
    if len(contact_points) == 0:
        raise RuntimeError(f"No interface contacts under cutoff for {ppi_pair_id}")

    k1 = pos_labels[contact_points].astype(int)
    k2 = r_contact[contact_points].astype(int)

    # Keep negative metadata for compatibility with records.npy schema.
    kdt_v1 = cKDTree(v1_candidates)
    dneg, _ = kdt_v1.query(v2_all)
    k_neg2 = np.where(dneg > interface_cutoff)[0].astype(int)

    rec = {
        "ppi_pair_id": ppi_pair_id,
        "pdb_id": pdb_id,
        "split": split,
        "in_dir": str(in_dir),
        "k1": k1,
        "k2": k2,
        "k_neg2": k_neg2,
        "within_dneg": dneg[k_neg2].astype(float),
    }

    print(f"Built record for {ppi_pair_id}")
    print(f"  SC-filtered p1 candidates: {len(pos_labels)}")
    print(f"  positive contact pairs: {len(k1)}")
    print(f"  within negatives: {len(k_neg2)}")
    print(f"  first 5 pairs: {list(zip(k1[:5], k2[:5]))}")
    return rec

# Function to extract model_type from complex_id (e.g. "1ATP-PP_L_R" -> "PP")
def get_model_type(complex_id):
    return complex_id.split("-")[1].split("_")[0]


In [ ]:
# Check if the mode type is PP
complex_id = small_list[0]
model_type = get_model_type(complex_id)

# Proceed only if model_type is not PP
if model_type != "PP":
    PP_complex_id = complex_id.replace(f"-{model_type}_", "-PP_")
    pp_record = build_pp_record(PP_complex_id, split="train")
    pp_record
else:
    print(f"Logic for PP model {complex_id} is not yet implemented")


In [ ]:
# Guard check: verify required precomputation, mesh and pdb inputs for all complexes.
for cid in [PP_complex_id, complex_id]:
    assert_inputs_exist(cid)
print("File checks passed")

___
### Step 3 - Visualize the positive pairs on PP structure by py3Dmol

- Load the structures to py3Dmol view, p1 in cyan and p2 in pink
- Get the coordinates of vertices in k1 and k2, and add them to the py3Dmol view as small spheres (blue and red)
- For each pair of pathces, draw a thin green line between the centre coordinates of the two patches. 

In [ ]:
def load_chain_vertices(ppi_pair_id, pid):
    pdb_id, ch1, ch2 = parse_ppi_pair_id(ppi_pair_id)
    chain = ch1 if pid == "p1" else ch2
    ply_fn = PLY_DIR / f"{pdb_id}_{chain}.ply"
    return load_vertices_from_ply(ply_fn)


def get_patch_coords(ppi_pair_id, pid, vix, full_patch=False):
    """Get patch center (and optionally full patch) coordinates in original frame."""
    vertices = load_chain_vertices(ppi_pair_id, pid)
    center = vertices[vix]
    if not full_patch:
        return center, None

    list_idx_fn = PRECOMP_DIR / ppi_pair_id / f"{pid}_list_indices.npy"
    if not list_idx_fn.exists():
        return center, None
    neigh = np.load(list_idx_fn, allow_pickle=True)[vix]
    patch_coords = vertices[neigh]
    return center, patch_coords


def get_coords_from_indices(ppi_pair_id, pid, indices):
    vertices = load_chain_vertices(ppi_pair_id, pid)
    return vertices[np.asarray(indices, dtype=int)]


def nearest_vertex_indices(src_coords, dst_vertices):
    kdt = cKDTree(dst_vertices)
    d, idx = kdt.query(np.asarray(src_coords), k=1)
    return np.asarray(idx, dtype=int), np.asarray(d, dtype=float)


def load_pdb_structure(ppi_pair_id):
    parser = PDBParser(QUIET=True)
    pdb_path = PINDER_PDB_DIR / f"{ppi_pair_id}.pdb"
    return parser.get_structure(ppi_pair_id, str(pdb_path))


def add_struct_to_py3dmol(structure, view=None):
    from io import StringIO

    io = PDBIO()
    io.set_structure(structure)
    buf = StringIO()
    io.save(buf)
    if view is None:
        view = py3Dmol.view(width=900, height=650)
    view.addModel(buf.getvalue(), "pdb")
    return view


def add_spheres(view, coords, color, radius=0.3):
    if coords is None:
        return
    for pt in np.atleast_2d(coords):
        view.addSphere({"center": {"x": float(pt[0]), "y": float(pt[1]), "z": float(pt[2])}, "radius": float(radius), "color": color})


def show_pair_lines(view, coords1, coords2, color="green", radius=0.05):
    for a, b in zip(np.atleast_2d(coords1), np.atleast_2d(coords2)):
        view.addCylinder(
            {
                "start": {"x": float(a[0]), "y": float(a[1]), "z": float(a[2])},
                "end": {"x": float(b[0]), "y": float(b[1]), "z": float(b[2])},
                "radius": float(radius),
                "color": color,
            }
        )


def summarize_distances(name, d):
    d = np.asarray(d, dtype=float)
    print(
        f"{name}: n={len(d)}, min={d.min():.3f}, median={np.median(d):.3f}, "
        f"p95={np.percentile(d, 95):.3f}, max={d.max():.3f}"
    )


In [ ]:
# Step 3 execution: visualize positive pairs on PP.
max_pairs_to_draw = 250

pp_k1 = np.asarray(pp_record["k1"], dtype=int)
pp_k2 = np.asarray(pp_record["k2"], dtype=int)
pp_coords_p1 = get_coords_from_indices(PP_complex_id, "p1", pp_k1)
pp_coords_p2 = get_coords_from_indices(PP_complex_id, "p2", pp_k2)

_, pp_ch1, pp_ch2 = parse_ppi_pair_id(PP_complex_id)
n_draw = min(max_pairs_to_draw, len(pp_k1))

pp_view = add_struct_to_py3dmol(load_pdb_structure(PP_complex_id))
pp_view.setStyle({"chain": pp_ch1}, {"cartoon": {"color": "cyan"}})
pp_view.setStyle({"chain": pp_ch2}, {"cartoon": {"color": "pink"}})

add_spheres(pp_view, pp_coords_p1[:n_draw], "blue", radius=0.28)
add_spheres(pp_view, pp_coords_p2[:n_draw], "red", radius=0.28)
show_pair_lines(pp_view, pp_coords_p1[:n_draw], pp_coords_p2[:n_draw], color="green", radius=0.03)
pp_view.zoomTo()
print(f"Rendering {n_draw}/{len(pp_k1)} PP positive pairs")
pp_view

___
### Step 4 - Map the positive patch coordinates onto the equivalent chain in the complex containing AF2 model
- Load PP structure into py3Dmol of from both PP structure (cyan) and AP structure (pink)
- Add spheres for each interface patch coordinate: PP structure in blue and AP structure in red
- Create two side-by-side views for each of L chain and R chain

In [ ]:
# Step 4 execution: map PP positives (p1/L and p2/R) onto AP/PA models by nearest center.
from io import StringIO

def _to_pdb_block(structure):
    io = PDBIO()
    io.set_structure(structure)
    buf = StringIO()
    io.save(buf)
    return buf.getvalue()

max_pairs_to_draw_ap = 250
_, pp_ch1, pp_ch2 = parse_ppi_pair_id(PP_complex_id)
_, ap_ch1, ap_ch2 = parse_ppi_pair_id(complex_id)

pp_structure = load_pdb_structure(PP_complex_id)
ap_structure = load_pdb_structure(complex_id)
pp_block = _to_pdb_block(pp_structure)
ap_block = _to_pdb_block(ap_structure)

ap_map_view = py3Dmol.view(width=1400, height=650, viewergrid=(1, 2), linked=True)

chain_jobs = [
    ((0, 0), "p1", pp_coords_p1, pp_ch1, ap_ch1, "L chain (p1)"),
    ((0, 1), "p2", pp_coords_p2, pp_ch2, ap_ch2, "R chain (p2)"),
]

for panel, pid, pp_coords, pp_chain, ap_chain, title in chain_jobs:
    ap_vertices = load_chain_vertices(complex_id, pid)
    k_ap, d_ap = nearest_vertex_indices(pp_coords, ap_vertices)
    ap_coords = ap_vertices[k_ap]
    n_draw_ap = min(max_pairs_to_draw_ap, len(k_ap))

    summarize_distances(f"PP {pid} -> AP {pid} nearest-center distance", d_ap)
    print(f"Mapped {pid} indices AP: {len(k_ap)}")

    ap_map_view.addModel(pp_block, "pdb", viewer=panel)
    ap_map_view.addModel(ap_block, "pdb", viewer=panel)
    ap_map_view.setStyle({"model": 0, "chain": pp_chain}, {"cartoon": {"color": "cyan"}}, viewer=panel)
    ap_map_view.setStyle({"model": 1, "chain": ap_chain}, {"cartoon": {"color": "pink"}}, viewer=panel)
    ap_map_view.addLabel(title, {"fontColor": "black", "backgroundOpacity": 0.0, "fontSize": 14}, viewer=panel)

    for pt in np.atleast_2d(pp_coords[:n_draw_ap]):
        ap_map_view.addSphere(
            {"center": {"x": float(pt[0]), "y": float(pt[1]), "z": float(pt[2])}, "radius": 0.24, "color": "blue"},
            viewer=panel,
        )
    for pt in np.atleast_2d(ap_coords[:n_draw_ap]):
        ap_map_view.addSphere(
            {"center": {"x": float(pt[0]), "y": float(pt[1]), "z": float(pt[2])}, "radius": 0.24, "color": "red"},
            viewer=panel,
        )
    for a, b in zip(np.atleast_2d(pp_coords[:n_draw_ap]), np.atleast_2d(ap_coords[:n_draw_ap])):
        ap_map_view.addCylinder(
            {
                "start": {"x": float(a[0]), "y": float(a[1]), "z": float(a[2])},
                "end": {"x": float(b[0]), "y": float(b[1]), "z": float(b[2])},
                "radius": 0.025,
                "color": "green",
            },
            viewer=panel,
        )
    ap_map_view.zoomTo(viewer=panel)

ap_map_view

# TODO: create a record for the complex_id
# Build mapped record for complex_id in the same schema as build_pp_record().
# Here we map PP positive centers (p1/p2) onto target complex (AP/PA) by nearest vertex index.

target_p1_vertices = load_chain_vertices(complex_id, "p1")
target_p2_vertices = load_chain_vertices(complex_id, "p2")

k1_map, d1_map = nearest_vertex_indices(pp_coords_p1, target_p1_vertices)
k2_map, d2_map = nearest_vertex_indices(pp_coords_p2, target_p2_vertices)

summarize_distances("PP p1 -> target p1 nearest-center distance", d1_map)
summarize_distances("PP p2 -> target p2 nearest-center distance", d2_map)

# Recompute within negatives (k_neg2) in target complex, same logic as build_pp_record:
# query each p2 vertex against mapped positive p1 centers.
v1_candidates = target_p1_vertices[k1_map]
kdt_v1 = cKDTree(v1_candidates)
dneg, _ = kdt_v1.query(target_p2_vertices)
k_neg2 = np.where(dneg > POS_INTERFACE_CUTOFF)[0].astype(int)

pdb_id, _, _ = parse_ppi_pair_id(complex_id)
target_record = {
    "ppi_pair_id": complex_id,
    "pdb_id": pdb_id,
    "split": "train",
    "in_dir": str(PRECOMP_DIR / complex_id),
    "k1": k1_map.astype(int),
    "k2": k2_map.astype(int),
    "k_neg2": k_neg2,
    "within_dneg": dneg[k_neg2].astype(float),
}

print(f"Built mapped record for {complex_id}")
print(f"  mapped positive pairs: {len(target_record['k1'])}")
print(f"  within negatives: {len(target_record['k_neg2'])}")
print(f"  first 5 pairs: {list(zip(target_record['k1'][:5], target_record['k2'][:5]))}")




In [ ]:
# Step 4 execution: build mapped record for AP/PA and visualize PP <-> target correspondence.
from io import StringIO

def _to_pdb_block(structure):
    io = PDBIO()
    io.set_structure(structure)
    buf = StringIO()
    io.save(buf)
    return buf.getvalue()

def build_mapped_record_from_pp(pp_record, complex_id, split="train", interface_cutoff=POS_INTERFACE_CUTOFF):
    """
    Build a record in build_pp_record schema for a target complex by mapping
    PP positive patch centers (k1/k2) to nearest vertices on target p1/p2.
    """
    # PP positive coordinates (from PP record indices)
    pp_p1_vertices = load_chain_vertices(PP_complex_id, "p1")
    pp_p2_vertices = load_chain_vertices(PP_complex_id, "p2")
    pp_coords_p1 = pp_p1_vertices[pp_record["k1"]]
    pp_coords_p2 = pp_p2_vertices[pp_record["k2"]]

    # Target mesh vertices
    target_p1_vertices = load_chain_vertices(complex_id, "p1")
    target_p2_vertices = load_chain_vertices(complex_id, "p2")

    # Nearest-center mapping onto target
    k1_map, d1_map = nearest_vertex_indices(pp_coords_p1, target_p1_vertices)
    k2_map, d2_map = nearest_vertex_indices(pp_coords_p2, target_p2_vertices)

    summarize_distances(f"PP p1 -> {complex_id} p1 nearest-center distance", d1_map)
    summarize_distances(f"PP p2 -> {complex_id} p2 nearest-center distance", d2_map)

    # Within-negatives, same schema/logic as build_pp_record
    v1_candidates = target_p1_vertices[k1_map]
    kdt_v1 = cKDTree(v1_candidates)
    dneg, _ = kdt_v1.query(target_p2_vertices)
    k_neg2 = np.where(dneg > interface_cutoff)[0].astype(int)

    pdb_id, _, _ = parse_ppi_pair_id(complex_id)
    rec = {
        "ppi_pair_id": complex_id,
        "pdb_id": pdb_id,
        "split": split,
        "in_dir": str(PRECOMP_DIR / complex_id),
        "k1": k1_map.astype(int),
        "k2": k2_map.astype(int),
        "k_neg2": k_neg2,
        "within_dneg": dneg[k_neg2].astype(float),
    }

    return rec

# ---- Usage ----
mapped_record = build_mapped_record_from_pp(pp_record, complex_id, split="train")
mapped_record

In [ ]:
# --- Visualize the positive patch mapping results ---
def visualize_pp_to_target_correspondence(pp_record, mapped_record, max_pairs_to_draw=250):
    """
    Side-by-side visualization:
      left panel  = p1 (L chain): PP vs target mapped points
      right panel = p2 (R chain): PP vs target mapped points
    """
    pp_id = pp_record["ppi_pair_id"]
    target_id = mapped_record["ppi_pair_id"]

    _, pp_ch1, pp_ch2 = parse_ppi_pair_id(pp_id)
    _, tg_ch1, tg_ch2 = parse_ppi_pair_id(target_id)

    pp_structure = load_pdb_structure(pp_id)
    tg_structure = load_pdb_structure(target_id)
    pp_block = _to_pdb_block(pp_structure)
    tg_block = _to_pdb_block(tg_structure)

    # Coordinates from record indices
    pp_p1_vertices = load_chain_vertices(pp_id, "p1")
    pp_p2_vertices = load_chain_vertices(pp_id, "p2")
    tg_p1_vertices = load_chain_vertices(target_id, "p1")
    tg_p2_vertices = load_chain_vertices(target_id, "p2")

    pp_coords_p1 = pp_p1_vertices[pp_record["k1"]]
    pp_coords_p2 = pp_p2_vertices[pp_record["k2"]]
    tg_coords_p1 = tg_p1_vertices[mapped_record["k1"]]
    tg_coords_p2 = tg_p2_vertices[mapped_record["k2"]]

    n_draw_p1 = min(max_pairs_to_draw, len(mapped_record["k1"]))
    n_draw_p2 = min(max_pairs_to_draw, len(mapped_record["k2"]))

    view = py3Dmol.view(width=1400, height=650, viewergrid=(1, 2), linked=True)

    chain_jobs = [
        ((0, 0), pp_ch1, tg_ch1, "L chain (p1)", pp_coords_p1[:n_draw_p1], tg_coords_p1[:n_draw_p1]),
        ((0, 1), pp_ch2, tg_ch2, "R chain (p2)", pp_coords_p2[:n_draw_p2], tg_coords_p2[:n_draw_p2]),
    ]

    for panel, pp_chain, tg_chain, title, pp_pts, tg_pts in chain_jobs:
        view.addModel(pp_block, "pdb", viewer=panel)
        view.addModel(tg_block, "pdb", viewer=panel)
        view.setStyle({"model": 0, "chain": pp_chain}, {"cartoon": {"color": "cyan"}}, viewer=panel)
        view.setStyle({"model": 1, "chain": tg_chain}, {"cartoon": {"color": "pink"}}, viewer=panel)
        view.addLabel(title, {"fontColor": "black", "backgroundOpacity": 0.0, "fontSize": 14}, viewer=panel)

        for pt in np.atleast_2d(pp_pts):
            view.addSphere(
                {"center": {"x": float(pt[0]), "y": float(pt[1]), "z": float(pt[2])}, "radius": 0.24, "color": "blue"},
                viewer=panel,
            )
        for pt in np.atleast_2d(tg_pts):
            view.addSphere(
                {"center": {"x": float(pt[0]), "y": float(pt[1]), "z": float(pt[2])}, "radius": 0.24, "color": "red"},
                viewer=panel,
            )
        for a, b in zip(np.atleast_2d(pp_pts), np.atleast_2d(tg_pts)):
            view.addCylinder(
                {
                    "start": {"x": float(a[0]), "y": float(a[1]), "z": float(a[2])},
                    "end": {"x": float(b[0]), "y": float(b[1]), "z": float(b[2])},
                    "radius": 0.025,
                    "color": "green",
                },
                viewer=panel,
            )
        view.zoomTo(viewer=panel)

    return view

# ---- Use helper + visualize ----
ap_map_view = visualize_pp_to_target_correspondence(pp_record, mapped_record, max_pairs_to_draw=250)
ap_map_view